

---

### **Problem Statement**

Organizations often receive large and complex Excel files containing multiple sheets of structured or semi-structured data. Manually exploring these files to understand column types, identify missing values, detect outliers, analyze categorical and date fields, and generate meaningful summaries can be time-consuming and error-prone.

The objective is to develop an **automated Excel Insights Engine** that can:

* Load Excel files with multiple sheets.
* Provide essential metadata such as row and column counts, column names, and data types.
* Detect and quantify missing values.
* Generate summary statistics for numeric columns.
* Identify strong correlations between numeric variables.
* Detect outliers using statistical methods like IQR or Z-score.
* Analyze categorical columns for frequency distributions and data quality.
* Identify and summarize patterns in date columns, including day, month, and year distributions.
* Automatically create visualizations to help users interpret the data.
* Compile all findings into a structured report in markdown format or plain text.

This tool will be useful for **data analysts, auditors, business users, and domain experts** who need to extract quick insights from Excel data without writing manual code every time.

---


In [1]:
import os # for having access to the functionality of the operating system
import pandas as pd # for data manipulation and analysis
import numpy as np # for numerical operations
import matplotlib.pyplot as plt # for plotting graphs
from datetime import datetime # for manipulating dates and times

In [2]:
class ExcelInsights:
    """
    A class to analyze and visualize Excel data and generate insights.
    """

    def __init__(self, file_path=None):
        """
        Initializes the ExcelInsights class with a file path.
        
        Args:
            file_path (str, optional): The path to the Excel file. Defaults to None.
        """

        self.file_path = file_path # This path will later be used to load the Excel file.
        self.data = None # This will store the default DataFrame (usually the first sheet).
        self.sheets = {} # Allows multi-sheet Excel files to be handled easily and individually.
        self.insights = {} # Central store for all computed insights, useful for generating a final report.

    def load_excel(self, file_path=None):
        """
        Load an Excel file into pandas DataFrames.
    
        Args:
            file_path (str, optional): Path to the Excel file to load.
        
        Returns:
            bool: True if successful, False otherwise.
        """
        if file_path: # If a file path is provided, use it.
            self.file_path = file_path
        if not self.file_path: # If no file path is set, return False.
            print("Error: No file path provided.")
            return False

        try:
            # Load all sheets from Excel file
            excel_file = pd.ExcelFile(self.file_path) # Load the Excel file
            sheet_names = excel_file.sheet_names # Get all sheet names into a list
            # Print the names of the sheets found in the Excel file.
            print(f"Sheets found: {sheet_names}")

            # Store each sheet as a DataFrame in the sheets dictionary.
            for sheet in sheet_names: # ['Sales', 'Employees', 'Customers']
                self.sheets[sheet] = pd.read_excel(self.file_path, sheet_name=sheet)
                print(f"Loaded sheet: {sheet} with {len(self.sheets[sheet])} rows and {len(self.sheets[sheet].columns)} columns.")

            # Set the default data to the first sheet loaded.
            if sheet_names: # If there are any sheets, set the first one as the default data.
                self.data = self.sheets[sheet_names[0]] # Set the first sheet as the default data.
            print(f"Successfully loaded {len(sheet_names)} from {os.path.basename(self.file_path)}.")
            return True # Return True if the file was loaded successfully.
        except Exception as e: # If an error occurs, print it and return False.
            print(f"Error loading Excel file: {e}")
            return False

    def get_basic_info(self):
        """
        Get basic information about the loaded data.
        
        Returns:
            dict: Dictionary containing basic information about the data.
        """
        if self.data is None: # If no data is loaded, return an empty dictionary.
            print("No data loaded. Please load an Excel file first.")
            return {} # Return an empty dictionary if no data is loaded.
        # Get basic information about the DataFrame.
        info = {
            "rows": len(self.data),
            "columns": len(self.data.columns),
            "column_names": list(self.data.columns),
            "data_types": {col: str(dtype) for col, dtype in self.data.dtypes.items()},
            "missing_values": self.data.isnull().sum().to_dict(),
            "sheets": list(self.sheets.keys())
        }
        
        self.insights["basic_info"] = info
        return info

    def generate_summary_statistics(self, sheet_name=None):
        """
        Generate summary statistics for the specified sheet or the default data.
        
        Args:
            sheet_name (str, optional): The name of the sheet to analyze. Defaults to None.

        Returns:
            dict: Dictionary containing summary statistics for each numeric column.
        """
        if sheet_name: # If a sheet name is provided, use it.
            if sheet_name not in self.sheets: # Check if the sheet exists.
                print(f"Sheet '{sheet_name}' not found.")
                return {}
            data_to_analyze = self.sheets[sheet_name] # Use the specified sheet.
        else:
            if self.data is None: # If no data is loaded, return an empty dictionary.
                print("No data loaded. Please load an Excel file first.")
                return {}
            data_to_analyze = self.data # Use the default data.

        summary_stats = {} # Dictionary to hold summary statistics for each numeric column.
        numeric_columns = data_to_analyze.select_dtypes(include=[np.number]).columns # Get numeric columns.

        for col in numeric_columns: # For each numeric column, compute summary statistics.
            summary_stats[col] = {
                "mean": data_to_analyze[col].mean(),
                "median": data_to_analyze[col].median(),
                "std_dev": data_to_analyze[col].std(),
                "min": data_to_analyze[col].min(),
                "max": data_to_analyze[col].max(),
                "missing_values": data_to_analyze[col].isnull().sum()
            }

        self.insights[f"summary_statistics_{sheet_name or 'default'}"] = summary_stats
        return summary_stats

    def find_correlations(self, sheet_name=None, threshold=0.5):
        """
        Find correlations between numeric columns in the specified sheet or the default data.
        
        Args:
            sheet_name (str, optional): The name of the sheet to analyze. Defaults to None.
            threshold (float, optional): The correlation threshold to filter results. Defaults to 0.5.
        """
       if sheet_name and sheet_name in self.sheets:
            data = self.sheets[sheet_name]
        elif self.data is not None:
            data = self.data
        else:
            print("No data loaded. Please load an Excel file first.")
            return {}

    numerical_cols = data.select_dtypes(include=[np.number]).columns.tolist()
    if not numerical_cols:
        print("No numeric columns found for correlation analysis.")
        return {}   

    correlation_matrix = data[numerical_cols].corr().abs() # Compute the absolute correlation matrix
    high_correlation_pairs = {} # Dictionary to hold pairs of columns with high correlation
    for i in range(len(correlation_matrix.columns)):
        for j in range(i + 1, len(correlation_matrix.columns)):
           col1 = correlation_matrix.columns[i]
           print(col1)
           col2 = correlation_matrix.columns[j]
           print(col2)
           corr_value = correlation_matrix.iloc[col1, col2]

           if corr_value >= threshold:
               high_correlation_pairs[f"{col1} - {col2}"] = corr_value



        if "correlation_analysis" not in self.insights:
            self.insights["correlation_analysis"] = {}
            if sheet_name:
                self.insights["correlation_analysis"][sheet_name] = high_correlation_pairs
            else:
                self.insights["correlation_analysis"]["default"] = high_correlation_pairs
            return high_correlation_pairs
        
           

IndentationError: unindent does not match any outer indentation level (<string>, line 124)

In [3]:
def find_correlations(self, sheet_name=None, threshold=0.5):
    """Find highly correlated numeric columns in a sheet or the default data."""
    if sheet_name and sheet_name in self.sheets:
        data = self.sheets[sheet_name]
    elif self.data is not None:
        data = self.data
    else:
        print("No data loaded. Please load an Excel file first.")
        return {}

    numerical_cols = data.select_dtypes(include=[np.number]).columns.tolist()
    if not numerical_cols:
        print("No numeric columns found for correlation analysis.")
        return {}

    correlation_matrix = data[numerical_cols].corr().abs()
    high_correlation_pairs = {}

    for i in range(len(correlation_matrix.columns)):
        for j in range(i + 1, len(correlation_matrix.columns)):
            col1 = correlation_matrix.columns[i]
            col2 = correlation_matrix.columns[j]
            corr_value = correlation_matrix.iloc[i, j]

            if corr_value >= threshold:
                high_correlation_pairs[f"{col1} - {col2}"] = corr_value

    if "correlation_analysis" not in self.insights:
        self.insights["correlation_analysis"] = {}

    result_key = sheet_name or "default"
    self.insights["correlation_analysis"][result_key] = high_correlation_pairs

    return high_correlation_pairs

In [4]:
# Create an instance of the class ExcelInsights
file_path = "samples/sample_data.xlsx"  # Path to the Excel file
excel_insights = ExcelInsights(file_path)
# Call load_excel() method to load the Excel file
excel_insights.load_excel()
print("\n\nBasic Information:")
# Call get_basic_info() method to get basic information about the loaded data
basic_info = excel_insights.get_basic_info()
# Print the basic information
for key, value in basic_info.items():
    print(f"{key}: {value}")
print("\n\nSummary Statistics:")
# Call generate_summary_statistics() method to get summary statistics for the default data
summary_stats = excel_insights.generate_summary_statistics()
# Print the summary statistics
for col, stats in summary_stats.items():
    print(f"\n{col}:")
    for stat_name, stat_value in stats.items():
        print(f"  {stat_name}: {stat_value}")

print("\n\nCorrelation Analysis:")
# Call find_correlations() method to find correlations between numeric columns
correlation_results = excel_insights.find_correlations(threshold=0.5)
# print the correlation
for keyvalue, corr_value in correlation_results.items():
    print(f"{keyvalue}: {corr_value}")

NameError: name 'ExcelInsights' is not defined